# Comparativa RAG V2 — Experimentos Factoriales (banco ampliado)

Este notebook presenta los resultados de los **4 experimentos comparativos** que justifican las decisiones de arquitectura del pipeline RAG V2.

**Banco de evaluación**: 72 casos (expected_pillars ≠ ∅, crisis_expected ≠ HIGH). La emoción de cada caso se inyecta directamente al routing sin invocar clasificador externo.  
**Evaluación**: todas las configuraciones se evalúan sobre el **conjunto completo de evaluación** (72 casos).

## 0. Imports y configuración

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import numpy as np
import pandas as pd

%matplotlib inline
plt.rcParams.update({
    "figure.dpi": 120, "font.size": 11,
    "axes.spines.top": False, "axes.spines.right": False,
})

RESULTS_DIR = Path('../results/rag_experiments')

def load_json(fname: str) -> dict | None:
    p = RESULTS_DIR / fname
    if not p.exists():
        print(f'[AVISO] {fname} no encontrado. Ejecuta primero el script correspondiente.')
        return None
    return json.loads(p.read_text(encoding='utf-8'))

print('Imports OK.')

## Experimento 1 — Estrategia de Chunking

Compara el **chunking manual terapéutico** (A) con el **chunking automático** `RecursiveCharacterTextSplitter(300/50)` (B, Ref. V1).

> ⚠️ **Advertencia metodológica**: la métrica P@k favorece artificialmente al chunking automático porque 181 chunks > 40 aumenta la cobertura de pilar por azar. La estrategia A se elige por **integridad clínica**: cada chunk manual contiene una técnica terapéutica completa; el chunking automático puede cortar una técnica a la mitad.

In [ ]:
data_c = load_json('chunking_results.json')
if data_c:
    configs = [
        ('estrategia_a', 'A — Manual terapéutico'),
        ('estrategia_b', 'B — Automático (Ref. V1)'),
    ]
    rows = []
    for cfg_key, label in configs:
        m = data_c.get(cfg_key, {})
        rows.append({
            'Config': f'{label} ({m.get("n_chunks", "?")} chunks)',
            'P@1': m.get('precision_at_1_media', 0),
            'P@3': m.get('precision_at_3_media', 0),
            'Hit@Any': m.get('hit_at_any_rate', 0),
            'n': m.get('n', '?'),
        })
    df = pd.DataFrame(rows).set_index('Config')
    print(f'=== Exp.1 — Chunking (n={data_c.get("banco", {}).get("n_total", 72)}) ===')
    display(df.style.format({'P@1': '{:.3f}', 'P@3': '{:.3f}', 'Hit@Any': '{:.3f}'}))

    # Figura
    labels = ['A Manual', 'B Auto (Ref.V1)']
    p3 = [data_c.get(k, {}).get('precision_at_3_media', 0) for k in ('estrategia_a', 'estrategia_b')]
    hit = [data_c.get(k, {}).get('hit_at_any_rate', 0) for k in ('estrategia_a', 'estrategia_b')]
    n = data_c.get('banco', {}).get('n_total', '?')
    x = np.arange(2)
    fig, ax = plt.subplots(figsize=(6, 4))
    b1 = ax.bar(x-0.18, p3,  0.33, label='P@3',     color='#4CAF50', alpha=0.82)
    b2 = ax.bar(x+0.18, hit, 0.33, label='Hit@Any', color='#FF9800', alpha=0.82)
    for bars, vals in [(b1, p3), (b2, hit)]:
        for bar, v in zip(bars, vals):
            ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
                    f'{v:.2f}', ha='center', va='bottom', fontsize=9)
    ax.set_xticks(x); ax.set_xticklabels(labels)
    ax.set_ylim(0, 1.15); ax.set_ylabel('Métrica'); ax.legend(fontsize=9)
    ax.set_title(f'Exp.1 — Chunking manual vs. automático (n={n})', fontweight='bold')
    plt.tight_layout(); plt.show()
    print('[Nota] P@k no es criterio de selección. Ver integridad clínica en el script.')

## Experimento 2 — Modelo de Embeddings

Compara 4 modelos sobre el corpus manual (72 casos). El modelo `paraphrase-multilingual-mpnet` es la referencia V1.

In [ ]:
data_e = load_json('embedding_results.json')
if data_e:
    modelos = data_e.get('modelos_evaluados', [])
    rows = []
    for m in modelos:
        tag = ' [Ref.V1]' if m.get('es_referencia_v1') else ''
        rows.append({
            'Modelo': m['short_name'] + tag,
            'P@1': m.get('precision_at_1_media', 0),
            'P@3': m.get('precision_at_3_media', 0),
            'Hit@Any': m.get('hit_at_any_rate', 0),
            'lat_ms': m.get('latencia_media_ms', 0),
            'dim': m.get('dim', 0),
            'n': m.get('n', '?'),
        })
    df = pd.DataFrame(rows).set_index('Modelo')
    n = data_e.get('banco', {}).get('n_total', 72)
    print(f'=== Exp.2 — Modelos de embeddings (n={n}) ===')
    display(df.style.format({'P@1': '{:.3f}', 'P@3': '{:.3f}', 'Hit@Any': '{:.3f}', 'lat_ms': '{:.1f}'}))

    names = [m['short_name'].split('/')[-1][:20] for m in modelos]
    x = np.arange(len(modelos))
    fig, ax = plt.subplots(figsize=(9, 4))
    vals = [m.get('precision_at_3_media', 0) for m in modelos]
    bars = ax.bar(x, vals, color='#2196F3', alpha=0.82)
    for bar, v in zip(bars, vals):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
                f'{v:.3f}', ha='center', va='bottom', fontsize=9)
    ax.set_xticks(x); ax.set_xticklabels(names, fontsize=9, rotation=10)
    ax.set_ylim(0, 1.15); ax.set_ylabel('P@3')
    ax.set_title(f'Exp.2 — P@3 por modelo de embeddings (n={n})', fontweight='bold')
    plt.tight_layout(); plt.show()

## Experimento 3 — Base de Datos Vectorial

Compara FAISS (Ref. V1, sin routing) con ChromaDB en 3 modelos × 2 modos (sin/con routing emocional).


In [ ]:
data_v = load_json('vectorstore_results.json')
if data_v:
    key_labels = {
        'faiss_global':            'FAISS global (Ref. V1)',
        'chromadb_mpnet_nf':       'ChromaDB+mpnet, sin rout.',
        'chromadb_mpnet_routing':  'ChromaDB+mpnet, +routing',
        'chromadb_distil_nf':      'ChromaDB+distil, sin rout.',
        'chromadb_distil_routing': 'ChromaDB+distil, +routing',
        'chromadb_bge_nf':         'ChromaDB+bge, sin rout.',
        'chromadb_bge_routing':    'ChromaDB+bge, +routing',
    }
    rows = []
    for k, label in key_labels.items():
        m = data_v.get(k, {})
        rows.append({'Config': label,
                     'P@1': m.get('precision_at_1_media', 0),
                     'P@3': m.get('precision_at_3_media', 0),
                     'Hit': m.get('hit_at_any_rate', 0)})
    df = pd.DataFrame(rows).set_index('Config')
    n = data_v.get('banco', {}).get('n_total', 72)
    print(f'=== Exp.3 — Vectorstore (n={n}) ===')
    display(df.style.format('{:.3f}'))

    labels = list(key_labels.values())
    colors = ['#607D8B','#2196F3','#1565C0','#4CAF50','#2E7D32','#FF9800','#E65100']
    vals = [data_v.get(k, {}).get('precision_at_3_media', 0) for k in key_labels]
    fig, ax = plt.subplots(figsize=(9, 5))
    bars = ax.barh(labels, vals, color=colors, alpha=0.85)
    for bar, v in zip(bars, vals):
        ax.text(v+0.005, bar.get_y()+bar.get_height()/2, f'{v:.3f}', va='center', fontsize=9)
    ax.set_xlim(0, 1.1); ax.set_xlabel('P@3')
    ax.invert_yaxis()
    ax.set_title(f'Exp.3 — P@3 por configuración (n={n})', fontweight='bold')
    plt.tight_layout(); plt.show()

## Experimento 4 — Búsqueda Híbrida: Diseño Factorial 2×2

**Factor 1 (BM25)**: semántico puro vs. fusión min-max BM25+semántico  
**Factor 2 (Routing)**: sin routing vs. con routing emocional  

Todas las configuraciones se evalúan sobre el conjunto completo. Los **efectos factoriales** son descriptivos; no se declara ningún ganador.

In [ ]:
data_h = load_json('hybrid_search_results.json')
if data_h:
    cfg_map = {
        'configuracion_a': 'A — Sem. global (Ref.V1)',
        'configuracion_b': 'B — Híbrido global',
        'configuracion_c': 'C — Sem.+routing',
        'configuracion_d': 'D — Híbrido+routing',
    }
    rows = []
    for k, label in cfg_map.items():
        m = data_h.get(k, {})
        rows.append({'Config': label,
                     'P@1': m.get('precision_at_1_media', 0),
                     'P@3': m.get('precision_at_3_media', 0),
                     'Hit': m.get('hit_at_any_rate', 0),
                     'n': m.get('n', '?')})
    df = pd.DataFrame(rows).set_index('Config')
    n = data_h.get('banco', {}).get('n_total', 72)
    print(f'=== Exp.4 — Factorial 2×2 (n={n}) ===')
    display(df.style.format({'P@1': '{:.3f}', 'P@3': '{:.3f}', 'Hit': '{:.3f}'}))

    # Efectos factoriales
    ef = data_h.get('efectos_factoriales', {})
    print('\n=== Efectos factoriales (ΔP@3) — solo descriptivos ===')
    print(f'  B−A (BM25, sin routing):    {ef.get("bm25_sin_routing_B_minus_A", 0):+.3f}')
    print(f'  C−A (routing, sin BM25):    {ef.get("routing_sin_bm25_C_minus_A", 0):+.3f}')
    print(f'  D−C (BM25, con routing):    {ef.get("bm25_con_routing_D_minus_C", 0):+.3f}')
    print(f'  D−A (efecto total):         {ef.get("efecto_total_D_minus_A", 0):+.3f}')
    print('  Nota: C y D son próximas cuando el efecto BM25 con routing (D-C) es pequeño.')

    # Figura: P@3 y Hit@Any para las 4 configs
    cfg_keys = list(cfg_map.keys())
    cfg_labels = ['A', 'B', 'C', 'D']
    colors = ['#607D8B', '#2196F3', '#4CAF50', '#FF9800']
    p3 = [data_h.get(k, {}).get('precision_at_3_media', 0) for k in cfg_keys]
    hit = [data_h.get(k, {}).get('hit_at_any_rate', 0) for k in cfg_keys]
    x = np.arange(4)
    fig, ax = plt.subplots(figsize=(8, 4))
    b1 = ax.bar(x-0.2, p3,  0.35, label='P@3',     color=colors, alpha=0.82)
    b2 = ax.bar(x+0.2, hit, 0.35, label='Hit@Any', color=colors, alpha=0.45, hatch='//')
    for bars, vals in [(b1, p3), (b2, hit)]:
        for bar, v in zip(bars, vals):
            ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.008,
                    f'{v:.2f}', ha='center', va='bottom', fontsize=8)
    ax.set_xticks(x); ax.set_xticklabels(cfg_labels)
    ax.set_ylim(0, 1.15); ax.set_ylabel('Métrica')
    handles = [mpatches.Patch(color=c, label=l) for c, l in zip(colors, ['A','B','C','D'])]
    ax.legend(handles=handles, fontsize=9, title='Config')
    ax.set_title(f'Exp.4 — Factorial 2×2 BM25×routing (n={n})', fontweight='bold')
    plt.tight_layout(); plt.show()

---
## Resumen descriptivo

Este notebook muestra los resultados de los 4 experimentos RAG sin declarar ninguna configuración como ganadora. Los datos son insumos para la decisión del investigador.

**Consideraciones clave**:
- **Exp.1 (chunking)**: la decisión se basa en integridad clínica, no en P@k.
- **Exp.2 (embeddings)**: observar latencia vs. P@3 según las restricciones de producción.
- **Exp.3 (vectorstore)**: el efecto del routing domina; el tipo de vectorstore importa menos.
- **Exp.4 (búsqueda)**: comparar C vs. D para decidir si el BM25 aporta suficiente sobre el routing puro.

---
## Conclusión: configuración seleccionada para V2

A partir de la batería factorial 2×2 (BM25 × routing) y las comparativas de chunking, embeddings y vectorstore, se selecciona como configuración del motor RAG de V2 la combinación **ChromaDB + paraphrase-spanish-distilroberta + routing emocional, con recuperación semántica pura (sin BM25)** — la Config C del diseño factorial.

La justificación es la siguiente:

1. **El aporte teórico de BM25 no se materializa en este dominio de uso.** BM25 resulta ventajoso cuando la consulta contiene términos o entidades exactas que también aparecen en el documento (solapamiento léxico). Sin embargo, el usuario real de este sistema —un adolescente que describe su situación de ciberacoso— se expresa en lenguaje natural y emocional ("no sé a quién acudir", "me amenazan y tengo miedo"), no mediante entidades exactas. Los contactos y recursos concretos (teléfonos de ayuda, organismos) residen en el corpus, no en la consulta, por lo que recuperarlos es una tarea de correspondencia semántica, no léxica. El escenario donde BM25 sería superior no se produce en el uso real.

2. **Parsimonia.** Ante la ausencia de aporte empírico del componente léxico, se descarta BM25 por simplicidad arquitectónica, reduciendo la superficie de mantenimiento y facilitando la trazabilidad clínica de las recuperaciones.

En conjunto, los datos y el conocimiento del usuario objetivo convergen en una arquitectura de recuperación semántica en español con enrutamiento emocional, descartando la fusión híbrida tras su evaluación empírica.